# 5.1 — Tabela DE-PARA de materiais (Supersessão)

- **Propósito:** Relacionar cada material original ao seu código acabado em item principal de cadeia, considerando listas técnicas 1:1 e tratamento de sufixos especiais.
- **Entradas:** `pr_cadastrao.material_cadastrao` (cadastrão) · `/Volumes/parts_hdbk_sandbox/dm_materials/sap/bom_production_center/` (lista técnica)
- **Saída:** `pr_cadastrao.material_de_para`
- **Chave:** Empresa + Material · **Carga:** Completa

In [0]:
import re
import unicodedata
import uuid

from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# ---------------------------------------------------------------------------
# Parâmetros e constantes
# ---------------------------------------------------------------------------

CATALOG = "parts_hdbk_sandbox"
SCHEMA = "pr_cadastrao"

# Tabela fonte: cadastrão refinado
CADASTRAO_TABLE = f"{CATALOG}.{SCHEMA}.material_cadastrao"

# Volume fonte: lista técnica do centro produtivo
BOM_PATH = "/Volumes/parts_hdbk_sandbox/dm_materials/sap/bom_production_center/"

# Tabela destino
OUTPUT_TABLE = f"{CATALOG}.{SCHEMA}.material_de_para"

# Tipos de material elegíveis
TIPOS_MATERIAL_VALIDOS = ["ZHAW", "ZFER", "ZRO1"]

# Grupos de compradores que indicam montagem in-house
GRUPOS_COMPRADORES_EXCLUIR = ["040", "054"]

# Exportação de CSVs de validação (por padrão desativada)
EXPORTAR_VALIDACAO = False
VALIDACAO_DIR = "/Volumes/parts_hdbk_sandbox/pr_cadastrao/out_material_de_para"

In [0]:
def sanitize_col_name(name: str) -> str:
    """
    Sanitiza nomes de colunas: remove acentos, converte para lowercase,
    substitui caracteres inválidos por underscore.
    """
    if not name:
        return "col_unknown"
    nfkd = unicodedata.normalize("NFKD", name)
    ascii_name = nfkd.encode("ASCII", "ignore").decode("ASCII")
    clean = re.sub(r"[^a-zA-Z0-9_]", "_", ascii_name)
    clean = re.sub(r"_+", "_", clean).strip("_")
    return clean.lower()


def tratar_item_principal(col, fallback_col):
    """
    Substitui valores nulos, vazios ou 'SIM' pelo valor do campo de fallback.

    Args:
        col: Coluna contendo o item_principal_cadeia.
        fallback_col: Coluna a ser usada quando item_principal_cadeia for nulo/SIM.

    Returns:
        Coluna Spark com o tratamento aplicado.
    """
    return (
        F.when(
            col.isNull()
            | (F.trim(col) == "")
            | (F.upper(F.trim(col)) == "SIM"),
            fallback_col,
        ).otherwise(col)
    )

In [0]:
# ---------------------------------------------------------------------------
# 5.1.1 — Preparação da lista técnica
# ---------------------------------------------------------------------------

df_bom_raw = (
    spark.read.format("excel")
    .option("header", "false")
    .option("inferSchema", "false")
    .load(BOM_PATH)
)

# Extrair nomes de colunas da primeira linha (header embutido no dado)
header_row = df_bom_raw.first()
col_names = [sanitize_col_name(header_row[i] or f"col_{i}") for i in range(len(df_bom_raw.columns))]
df_bom = df_bom_raw.toDF(*col_names)

# Remover linhas de header duplicadas (cada arquivo .xlsx traz seu próprio header)
df_bom = df_bom.filter(F.col("material") != "Material")

# 5.1.1, passo 2: Criar campo empresa a partir do centro (2 primeiros dígitos + "00")
df_bom = df_bom.withColumn("empresa", F.concat(F.substring("centro", 1, 2), F.lit("00")))

# 5.1.1, passo 3: Remover materiais acabados com sufixos especiais (" C", " G", " NP")
df_bom = df_bom.filter(
    ~(
        F.col("material").rlike(r"\s+C$")
        | F.col("material").rlike(r"\s+G$")
        | F.col("material").rlike(r"\s+NP$")
    )
)

print(f"5.1.1 — BOM após preparação: {df_bom.count()} linhas")
display(df_bom.select("empresa", "centro", "material", "componente").limit(10))

In [0]:
# ---------------------------------------------------------------------------
# 5.1.2 — Tratamento do componente e seleção de relacionamentos 1:1
# ---------------------------------------------------------------------------

# Leitura do cadastrão com colunas necessárias
df_cad = spark.table(CADASTRAO_TABLE).select(
    "empresa", "material", "tipo_de_material", "grupo_de_compradores", "item_principal_cadeia"
)

# 5.1.2, passos 4 e 5: Join BOM com cadastrão para trazer tipo_de_material do Componente
# e manter somente tipos elegíveis (ZHAW, ZFER, ZRO1)
df_bom_comp = (
    df_bom.alias("bom")
    .join(
        df_cad.alias("cad_comp"),
        (F.col("bom.empresa") == F.col("cad_comp.empresa"))
        & (F.col("bom.componente") == F.col("cad_comp.material")),
        "inner",
    )
    .filter(F.col("cad_comp.tipo_de_material").isin(TIPOS_MATERIAL_VALIDOS))
    .select(
        F.col("bom.empresa"),
        F.col("bom.material"),
        F.col("bom.componente"),
        F.col("cad_comp.item_principal_cadeia").alias("item_principal_cadeia"),
    )
)

# Saída 5.1.2, passos 4 e 5 (chave do join: empresa + componente)
print(f"5.1.2, passos 4 e 5 — Join BOM × Cadastrão (tipos elegíveis): {df_bom_comp.count()}")
display(df_bom_comp.limit(10))

# 5.1.2, passos 6 e 7: Agrupar por empresa + material e manter apenas relações 1:1
# (materiais produzidos a partir de exatamente um componente)
w_count = Window.partitionBy("empresa", "material")
df_bom_1to1 = (
    df_bom_comp.withColumn("qtd_componentes", F.count("*").over(w_count))
    .filter(F.col("qtd_componentes") == 1)
    .drop("qtd_componentes")
)

# Saída 5.1.2, passos 6 e 7 (chave da partição: empresa + material)
# Preservar com 4 colunas antes de ser reduzido no passo 12
df_bom_1to1_passos_6_7 = df_bom_1to1

print(f"\n5.1.2, passos 6 e 7 — Relações 1:1: {df_bom_1to1.count()}")
display(df_bom_1to1.limit(10))

In [0]:
# ---------------------------------------------------------------------------
# 5.1.2 / 5.1.3 — Tratamento do Item Principal de Cadeia
# ---------------------------------------------------------------------------

# 5.1.2, passos 8 a 10: Converter componente em item principal de cadeia
# Quando item_principal_cadeia for nulo ou "SIM", usar o próprio componente
df_bom_1to1 = df_bom_1to1.withColumn(
    "item_principal_cadeia_componente",
    tratar_item_principal(F.col("item_principal_cadeia"), F.col("componente")),
)

print("5.1.2, passos 8 a 10 — Item principal do componente (chave: empresa + componente):")
display(df_bom_1to1.select("empresa", "componente", "item_principal_cadeia", "item_principal_cadeia_componente").limit(10))

# 5.1.2, passo 11: Remover linhas onde item_principal_cadeia_componente termina com " N"
# (materiais sem embalagem serão tratados diretamente na Etapa B)
df_bom_1to1 = df_bom_1to1.filter(~F.col("item_principal_cadeia_componente").rlike(r"\s+N$"))

print(f"\n5.1.2, passo 11 — Após remover sufixo ' N': {df_bom_1to1.count()} linhas")

# 5.1.2, passo 12: Manter somente os campos necessários
df_bom_1to1 = df_bom_1to1.select("empresa", "item_principal_cadeia_componente", "material")

print("\n5.1.2, passo 12 — Campos mantidos (chave: empresa + material):")
display(df_bom_1to1.limit(10))

# 5.1.3, passos 13 e 14: Join com cadastrão para trazer dados do material produzido
# Excluir grupos de compradores 040 e 054 (montagem in-house)
df_etapa_a = (
    df_bom_1to1.alias("bom1")
    .join(
        df_cad.alias("cad_mat"),
        (F.col("bom1.empresa") == F.col("cad_mat.empresa"))
        & (F.col("bom1.material") == F.col("cad_mat.material")),
        "inner",
    )
    .filter(~F.col("cad_mat.grupo_de_compradores").isin(GRUPOS_COMPRADORES_EXCLUIR))
    .select(
        F.col("bom1.empresa"),
        F.col("bom1.item_principal_cadeia_componente"),
        F.col("bom1.material"),
        F.col("cad_mat.item_principal_cadeia").alias("item_principal_cadeia"),
    )
)

# Saída 5.1.3, passos 13 e 14 (chave do join: empresa + material)
print(f"\n5.1.3, passos 13 e 14 — Join com cadastrão (excluídos grupos 040/054): {df_etapa_a.count()}")
display(df_etapa_a.limit(10))

# 5.1.3, passos 15 a 17: Converter material produzido em item principal de cadeia
df_etapa_a = df_etapa_a.withColumn(
    "item_principal_cadeia_material_produzido",
    tratar_item_principal(F.col("item_principal_cadeia"), F.col("material")),
)

# 5.1.3, passos 18 e 19: Manter campos finais e remover duplicatas
df_etapa_a = df_etapa_a.select(
    "empresa", "item_principal_cadeia_componente", "item_principal_cadeia_material_produzido"
).dropDuplicates()

print(f"\n5.1.3, passos 18 e 19 — Etapa A final (deduplicada): {df_etapa_a.count()} relações")
display(df_etapa_a.limit(10))

In [0]:
# ---------------------------------------------------------------------------
# 5.2.1 — Preparação e tratamento dos materiais
# ---------------------------------------------------------------------------

# 5.2.1, passos 1 e 2: Selecionar materiais com tipos elegíveis do cadastrão
df_depara = (
    spark.table(CADASTRAO_TABLE)
    .select("empresa", "material", "tipo_de_material", "grupo_de_compradores", "item_principal_cadeia")
    .filter(F.col("tipo_de_material").isin(TIPOS_MATERIAL_VALIDOS))
)

# 5.2.1, passo 3: Criar campo material_sem_cgn com remoção condicional de sufixos
# Regras:
#   " N"  → remover sempre
#   " C"  → remover sempre
#   " G"  → empresa 0500: remover | empresa 0200: remover se grupo_de_compradores <> "040"
#   " NP" → remover sempre
df_depara = df_depara.withColumn(
    "material_sem_cgn",
    F.when(
        F.col("material").rlike(r"\s+N$"),
        F.rtrim(F.regexp_replace(F.col("material"), r"\s+N$", "")),
    )
    .when(
        F.col("material").rlike(r"\s+C$"),
        F.rtrim(F.regexp_replace(F.col("material"), r"\s+C$", "")),
    )
    .when(
        (F.col("material").rlike(r"\s+G$")) & (F.col("empresa") == "0500"),
        F.rtrim(F.regexp_replace(F.col("material"), r"\s+G$", "")),
    )
    .when(
        (F.col("material").rlike(r"\s+G$"))
        & (F.col("empresa") == "0200")
        & (F.col("grupo_de_compradores") != "040"),
        F.rtrim(F.regexp_replace(F.col("material"), r"\s+G$", "")),
    )
    .when(
        F.col("material").rlike(r"\s+NP$"),
        F.rtrim(F.regexp_replace(F.col("material"), r"\s+NP$", "")),
    )
    .otherwise(F.col("material")),
)

# Preservar saída de 5.2.1 antes de ser sobrescrita na célula seguinte
df_depara_5_2_1 = df_depara

print(f"5.2.1 — Materiais elegíveis: {df_depara.count()}")
display(df_depara.select("empresa", "material", "material_sem_cgn").limit(10))

In [0]:
# ---------------------------------------------------------------------------
# 5.2.2 / 5.2.3 — Definição do item principal e associação com Etapa A
# ---------------------------------------------------------------------------

# 5.2.2, passos 4 a 7: Join com cadastrão para obter item principal do material_sem_cgn
df_depara = (
    df_depara.alias("dp")
    .join(
        df_cad.select("empresa", "material", "item_principal_cadeia").alias("cad_scgn"),
        (F.col("dp.empresa") == F.col("cad_scgn.empresa"))
        & (F.col("dp.material_sem_cgn") == F.col("cad_scgn.material")),
        "left",
    )
    .withColumn(
        "item_principal_cadeia_material_sem_cgn",
        tratar_item_principal(F.col("cad_scgn.item_principal_cadeia"), F.col("dp.material_sem_cgn")),
    )
    .select(
        F.col("dp.empresa"),
        F.col("dp.material"),
        F.col("dp.material_sem_cgn"),
        "item_principal_cadeia_material_sem_cgn",
    )
)

# Preservar saída de 5.2.2 antes de ser sobrescrita pelo join com Etapa A
df_depara_5_2_2 = df_depara

# Saída 5.2.2, passos 4 a 7 (chave do join: empresa + material_sem_cgn)
print("5.2.2, passos 4 a 7 — Item principal do material sem CGN (chave: empresa + material_sem_cgn):")
display(df_depara.limit(10))

# 5.2.3, passos 8 a 11: Left join com Etapa A para obter o material final
# Se não houver correspondência na Etapa A, usar o próprio item_principal_cadeia_material_sem_cgn
df_depara = (
    df_depara.alias("dp2")
    .join(
        df_etapa_a.alias("ea"),
        (F.col("dp2.empresa") == F.col("ea.empresa"))
        & (F.col("dp2.item_principal_cadeia_material_sem_cgn") == F.col("ea.item_principal_cadeia_componente")),
        "left",
    )
    .withColumn(
        "item_principal_cadeia_material_final",
        tratar_item_principal(
            F.col("ea.item_principal_cadeia_material_produzido"),
            F.col("dp2.item_principal_cadeia_material_sem_cgn"),
        ),
    )
    .select(
        F.col("dp2.empresa"),
        F.col("dp2.material"),
        F.col("dp2.material_sem_cgn"),
        F.col("dp2.item_principal_cadeia_material_sem_cgn"),
        "item_principal_cadeia_material_final",
    )
)
# Saída 5.2.3, passos 8 a 11 (chave do join: empresa + item_principal_cadeia_material_sem_cgn)
print(f"\n5.2.3, passos 8 a 11 — Tabela DE-PARA antes de deduplicar: {df_depara.count()} linhas")
display(df_depara.limit(10))

In [0]:
# ---------------------------------------------------------------------------
# Exibição dos itens extras gerados pelo fan-out da lista técnica
# Um componente pode ser insumo único de vários produtos acabados;
# o left join com a Etapa A expande essas linhas.
# ---------------------------------------------------------------------------

# Identificar materiais com mais de uma linha (fan-out)
w_dup = Window.partitionBy("empresa", "material")
df_com_contagem = df_depara.withColumn("_n", F.count("*").over(w_dup))

df_extras = (
    df_com_contagem
    .filter(F.col("_n") > 1)
    .drop("_n")
    .orderBy("empresa", "material", "item_principal_cadeia_material_final")
)

n_materiais = df_extras.select("empresa", "material").distinct().count()
n_linhas = df_extras.count()

print(f"Materiais com fan-out: {n_materiais}")
print(f"Linhas extras (serão removidas pelo dropDuplicates): {n_linhas - n_materiais}")
print(f"\nDetalhe — cada material abaixo aparece com múltiplos item_principal_cadeia_material_final:")
display(df_extras)

In [0]:
# ---------------------------------------------------------------------------
# Deduplicar: manter uma única linha por empresa + material
# ---------------------------------------------------------------------------
df_depara = df_depara.dropDuplicates(["empresa", "material"])

print(f"Tabela DE-PARA final: {df_depara.count()} linhas")
display(df_depara.limit(10))

In [0]:
# ---------------------------------------------------------------------------
# Exportação condicional de CSVs de validação
# Controlada pelo parâmetro EXPORTAR_VALIDACAO (célula 3)
# ---------------------------------------------------------------------------

if EXPORTAR_VALIDACAO:
    import os
    import zipfile
    from datetime import datetime

    os.makedirs(VALIDACAO_DIR, exist_ok=True)

    # Cada saída é preservada em variável própria nas células anteriores
    # para evitar perda por sobrescrita de df_depara e df_bom_1to1.
    exports = [
        (
            "01_5_1_1_lista_tecnica.csv",
            df_bom.select("empresa", "centro", "material", "componente"),
        ),
        (
            "02_5_1_2_passos_4_e_5_bom_cadastrao.csv",
            df_bom_comp,
        ),
        (
            "03_5_1_2_passos_6_e_7_relacoes_1to1.csv",
            df_bom_1to1_passos_6_7,
        ),
        (
            "04_5_1_2_passo_12_campos_finais.csv",
            df_bom_1to1,
        ),
        (
            "05_5_1_4_saida_etapa_a.csv",
            df_etapa_a,
        ),
        (
            "06_5_2_1_materiais_elegiveis.csv",
            df_depara_5_2_1.select(
                "empresa", "material", "tipo_de_material",
                "grupo_de_compradores", "material_sem_cgn",
            ),
        ),
        (
            "07_5_2_2_item_principal_sem_cgn.csv",
            df_depara_5_2_2,
        ),
        (
            "08_fan_out_itens_extras.csv",
            df_extras,
        ),
        (
            "09_5_2_4_tabela_de_para.csv",
            df_depara.select(
                "empresa", "material", "material_sem_cgn",
                "item_principal_cadeia_material_sem_cgn",
                "item_principal_cadeia_material_final",
            ),
        ),
    ]

    csv_paths = []
    for filename, df_export in exports:
        path = os.path.join(VALIDACAO_DIR, filename)
        df_export.toPandas().to_csv(path, index=False)
        rows = df_export.count()
        csv_paths.append(path)
        print(f"  ✓ {filename} ({rows:,} linhas)")

    # Zipar em /tmp (volumes UC não suportam escrita randômica do zipfile)
    import shutil
    import tempfile

    today = datetime.now().strftime("%Y%m%d")
    zip_name = f"validacao_de_para_{today}.zip"
    tmp_zip = os.path.join(tempfile.gettempdir(), zip_name)
    final_zip = os.path.join(VALIDACAO_DIR, zip_name)

    with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in csv_paths:
            zf.write(path, os.path.basename(path))

    # Copiar ZIP para o volume e limpar CSVs individuais
    shutil.copy2(tmp_zip, final_zip)
    os.remove(tmp_zip)
    for path in csv_paths:
        os.remove(path)

    zip_size_mb = os.path.getsize(final_zip) / (1024 * 1024)
    print(f"\nZIP criado: {final_zip} ({zip_size_mb:.1f} MB)")
    print(f"Arquivos incluídos: {len(csv_paths)}")
else:
    print("Exportação de validação desativada (EXPORTAR_VALIDACAO = False).")

In [0]:
# ---------------------------------------------------------------------------
# Persistência
# ---------------------------------------------------------------------------

CURRENT_USER = spark.sql("SELECT current_user()").first()[0]
LOAD_ID = str(uuid.uuid4())

df_final = df_depara.withColumns({
    "_ingested_at": F.from_utc_timestamp(F.current_timestamp(), "America/Sao_Paulo"),
    "_ingested_by": F.lit(CURRENT_USER),
    "_load_type": F.lit("full"),
    "_load_id": F.lit(LOAD_ID),
    "_source_file_path": F.lit(BOM_PATH),
})

spark.sql(f"DROP TABLE IF EXISTS {OUTPUT_TABLE}")
df_final.write.saveAsTable(OUTPUT_TABLE)

print(f"Tabela {OUTPUT_TABLE} criada com sucesso.")
print(f"Total de registros: {spark.table(OUTPUT_TABLE).count()}")
print(f"Load ID: {LOAD_ID}")

In [0]:
# ---------------------------------------------------------------------------
# Metadados e governança no Unity Catalog
# ---------------------------------------------------------------------------

METADATA_VERSION = "1"

props = (
    spark.sql(f"DESCRIBE DETAIL {OUTPUT_TABLE}")
    .select("properties").first()["properties"] or {}
)

if props.get("material_de_para_metadata_version") != METADATA_VERSION:
    # Comentário da tabela
    spark.sql(f"""
        COMMENT ON TABLE {OUTPUT_TABLE} IS
        'Tabela DE-PARA de materiais: relaciona cada material original ao seu codigo '
        'acabado em item principal de cadeia, considerando listas tecnicas 1:1 e '
        'tratamento de sufixos especiais (C, G, N, NP). '
        'Chave: empresa + material. Fonte: material_cadastrao + BOM centro produtivo.'
    """)

    # Comentários das colunas
    col_comments = {
        "empresa": "Codigo da empresa SAP (ex: 0200=2W, 0500=4W)",
        "material": "Codigo original do material no cadastrao",
        "material_sem_cgn": "Codigo do material apos remocao dos sufixos especiais (C, G, N, NP)",
        "item_principal_cadeia_material_sem_cgn": "Item principal da cadeia associado ao material sem sufixos",
        "item_principal_cadeia_material_final": "Codigo acabado final atribuido ao material, considerando relacoes 1:1 da lista tecnica",
        "_ingested_at": "Timestamp de ingestao (fuso America/Sao_Paulo)",
        "_ingested_by": "Usuario responsavel pela ingestao",
        "_load_type": "Tipo de carga (full/incremental)",
        "_load_id": "Identificador unico da execucao de carga",
        "_source_file_path": "Caminho do volume fonte",
    }
    for col_name, comment in col_comments.items():
        spark.sql(f"COMMENT ON COLUMN {OUTPUT_TABLE}.`{col_name}` IS '{comment}'")

    # Tags de governança
    spark.sql(f"""
        ALTER TABLE {OUTPUT_TABLE} SET TAGS (
            'domain' = 'materials',
            'layer' = 'refined',
            'source' = 'sap',
            'history_model' = 'full_overwrite',
            'data_classification' = 'internal'
        )
    """)

    # Propriedades de negócio
    spark.sql(f"""
        ALTER TABLE {OUTPUT_TABLE} SET TBLPROPERTIES (
            'business_owner' = 'Demand Planning',
            'technical_owner' = 'Parts Data Engineering',
            'data_domain' = 'materials',
            'source_system' = 'sap',
            'natural_key' = 'empresa,material',
            'refresh_frequency' = 'manual',
            'material_de_para_metadata_version' = '{METADATA_VERSION}'
        )
    """)

    print(f"Metadados aplicados (versão {METADATA_VERSION}).")
else:
    print(f"Metadados já estão na versão {METADATA_VERSION}. Nenhuma alteração necessária.")

In [0]:
# ---------------------------------------------------------------------------
# Validação final
# ---------------------------------------------------------------------------

df_result = spark.table(OUTPUT_TABLE)

total = df_result.count()
distinct_keys = df_result.select("empresa", "material").distinct().count()
duplicatas = total - distinct_keys

print(f"Total de registros: {total:,}")
print(f"Chaves distintas (empresa + material): {distinct_keys:,}")
print(f"Duplicatas: {duplicatas:,}")
print(f"\nDistribuição por empresa:")
df_result.groupBy("empresa").count().orderBy("empresa").show()

print("Amostra da tabela DE-PARA:")
display(
    df_result.select(
        "empresa", "material", "material_sem_cgn",
        "item_principal_cadeia_material_sem_cgn",
        "item_principal_cadeia_material_final",
    ).limit(20)
)